# Programmatic access to CDS services, an illustration with carbon stars
---

Manon Marchand¹, Anaïs Gonneau¹, Mark Allen¹, Ivan Brossard¹, Hendrik Heinl²

1. Université de Strasbourg, CNRS, Observatoire Astronomique de Strasbourg, UMR 7550, F-67000, Strasbourg, France
2. INAF-OATS, Italy

## Introduction

---

In this tutorial, we will see how the main [CDS](https://cds.unistra.fr/) services can be accessed programmatically via python.

We will also reproduce the results of Ripoche et al. 2020 ([2020MNRAS.495.2858R](https://simbad.cds.unistra.fr/simbad/sim-ref?bibcode=2020MNRAS.495.2858R))  where they study whether carbon stars can be reliable standard candles. They calibrate the distance measurement with the mean J band of the carbon-rich giant stars of the Magellanic Clouds.

More specifically, we will:
1. Gather information on a specific object using [![Simbad](https://custom-icon-badges.demolab.com/badge/Simbad-gray.svg?logo=simbad&logoColor=lightblue&logoWidth=20)](https://simbad.unistra.fr/simbad/ "https://simbad.unistra.fr/simbad/"), 
2. Visualize it through survey images using the iPy[![Aladin](https://custom-icon-badges.demolab.com/badge/Aladin-gray.svg?logo=aladin&logoColor=purple&logoWidth=20)](https://aladin.cds.unistra.fr/aladin.gml "https://aladin.cds.unistra.fr/aladin.gml") widget, 
3. Find and download a catalogue from [![Vizier](https://custom-icon-badges.demolab.com/badge/Vizier-gray.svg?logo=vizier&logoColor=orange&logoWidth=20)](https://vizier.cds.unistra.fr/viz-bin/VizieR "https://vizier.cds.unistra.fr/viz-bin/VizieR") and cross-match the sources with a large catalogue using [![Xmatch](https://custom-icon-badges.demolab.com/badge/Xmatch-gray.svg?logo=xmatch&logoColor=blue&logoWidth=20)](http://cdsxmatch.cds.unistra.fr/ "http://cdsxmatch.cds.unistra.fr/"). 

The use-case is on carbon stars, but of course you can design similar queries for your own research!


In [ ]:
# Standard python library
import json
from collections import Counter

# Astronomy tools
from astropy.coordinates import Angle, SkyCoord

# Access astronomical databases
from astroquery.simbad import Simbad
from astroquery.vizier import Vizier
from astroquery.xmatch import XMatch

# Sky visualization
from sidecar import Sidecar  # allows to open the widget in a new tab
from ipyaladin import Aladin

# MOC manipulation
from mocpy import MOC

# Plots
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from astropy.wcs import WCS

# Pyvo and registry
from pyvo import registry
from pyvo.dal.tap import TAPService

# Choose euro-vo registry endpoint
registry.choose_RegTAP_service("https://registry.euro-vo.org/regtap/tap")

## Step #1: Carbon stars in SIMBAD

[![Simbad](https://custom-icon-badges.demolab.com/badge/Simbad-gray.svg?logo=simbad&logoColor=lightblue&logoWidth=20)](https://simbad.unistra.fr/simbad/ "https://simbad.unistra.fr/simbad/")

---

SIMBAD is a compilation of the literature. It is edited each working day by a team of astronomers and documentalists.


![simbad](images/simbad.png)

### CW Leo in the SIMBAD database

Let's start our exploration of SIMBAD by gathering information on a single object: the star `CW Leo`.


In [ ]:
# Create an instance of the Simbad class
simbad = Simbad()

As with any python object, we can see the list of all the possible methods for SIMBAD by writing `simbad.` and then pressing `tab`:

![simbad_functions](images/methods.png) 

Or we can do it programmatically with `dir`

In [ ]:
dir(simbad)

In this whole list, let's use `query_object`. We can display its help page with:

In [ ]:
help(simbad.query_object)

We see that the method has to be called with the object name:

In [ ]:
first_object = simbad.query_object("CW Leo")
first_object

In [ ]:
## Exercise:

# In this cell, adapt the previous call to `query_object` to have a look at your own favorite object



By default, SIMBAD's `query_object` method returns mostly information about the coordinates of the queried object. One very important information is the `coo_bibcode` column. This provides the paper/catalogue (depending on the case) from which the SIMBAD team extracted the coordinates. Let's explore this paper/catalogue in more details:

In [ ]:
coo_bibcode = simbad.query_bibcode("2003yCat.2246....0C")
coo_bibcode

We see that the coordinates returned by `query_object` are taken from the 2MASS catalogue.

Another interesting thing: in the result of `query_object` we notice that the main name (`main_id` column) of the star is `IRC +10216`, not `CW Leo`. This is because the SIMBAD team also records every name given to an object in the literature. Let's print them: 

In [ ]:
cw_leo_list_of_names = simbad.query_objectids("CW Leo")
cw_leo_list_of_names.show_in_notebook(
    auto_fit_columns=True, selection_mode="cell"
)  # show in notebook can be useful to display longer astropy tables

At the moment we write this tutorial, the team has recorded 23 different names for `CW Leo`. This careful cross match is very convenient: we could retrieve information with the name `CW Leo` even if this is not the main identifier (`main_id`)!

Let's now personalize the output of `query_object` to learn more about this star. This is done by adding a field with the `add_votable_field` method. 

The full list of available fields can be printed with `list_votable_fields`.

In [ ]:
possible_fields = simbad.list_votable_fields()
possible_fields.show_in_notebook(auto_fit_columns=True, selection_mode="cell")

Let's ask for the object type and the spectral type of the star:

In [ ]:
simbad.add_votable_fields("otype", "sp_type")

In [ ]:
### Exercise:

# In this cell, add an other votable field of your interest



Now, when we call again `query_object`, we see that there are new columns in the output: `otype` and `sp_type`... and also the columns added by your own additional `votable_field`:

In [ ]:
cw_leo = simbad.query_object("CW Leo")
cw_leo

The object type appears as `C*`, meaning carbon star. This is a classification done by the SIMBAD team. 

Using the [SIMBAD's hierarchical classification of objects](https://simbad.cds.unistra.fr/guide/otypes.htx) (cf also Figure I-2), we learn that Carbon Stars (C*) are a sub-type of Evolved Stard (Ev*) which are themselved a sub-type of Stars (*). 

![object-types](images/object_types.png)

You can learn more on the features of the Simbad python module here: [https://astroquery.readthedocs.io/en/latest/simbad/simbad.html](https://astroquery.readthedocs.io/en/latest/simbad/simbad.html) .

### Advanced use of SIMBAD

*This section is for the eager astronomer, we will skip it in the live demonstration.*

In SIMBAD, the information is organized in [different tables](https://simbad.cds.unistra.fr/simbad/tap/tapsearch.html). The most powerful way to retrieve information from them is to use [ADQL (Astronomical Data Query Language)](https://simbad.cds.unistra.fr/simbad/tap/help/adqlHelp.html) in the `query_tap` method. 
You'll learn more about ADQL in another tutorial of this school.

In the previous part, we looked at  `ra`, `dec`, `otype`, and `sp_type`. These columns are all in the SIMBAD table of basic information, called `basic`.
The list of columns of the basic table can be printed with `list_columns`.

In [ ]:
basic_columns = simbad.list_columns("basic")
basic_columns.show_in_notebook(auto_fit_columns=True, selection_mode="cell")

But there is more information that you can find. It is in tables outside of `basic`. Let's list them: 

In [ ]:
list_tables = simbad.list_tables()
list_tables.show_in_notebook(auto_fit_columns=True, selection_mode="cell")

 For example, let's say that we are interested in the measurements of variability for our star. We have to write an ADQL query that joins the `basic` table to the `mesVar` table. 

 The method `list_linked_tables` can help build the join:

In [ ]:
linked_to_mes_var = simbad.list_linked_tables("mesVar")
linked_to_mes_var

![simbad-links](images/simbad-joins.png)

This helps us build the following ADQL query in which we link `basic` to `mesVar` by their common columns `oid` and `oidref`:

In [ ]:
variability_measurement = simbad.query_tap(
    """SELECT main_id, mesVar.*
    FROM basic join mesVar on oid = oidref
    WHERE main_id = 'IRC +10216'"""
)
variability_measurement

We see that this table records three papers in which the variability of `CW Leo` has been measured.

In [ ]:
## Exercise

# We can now write fun queries about carbon stars:

In [ ]:
# Q1 - Which carbon star has the most citations? (You'll recognise it ;) )

In [ ]:
### [SOLUTION Q1]

adql = """
SELECT TOP 1 main_id FROM basic WHERE otype = 'C*' ORDER BY nbref DESC
"""  # note: we sorted the results for the first time

In [ ]:
# Q2 - How many carbon stars are in SIMBAD right now? (It was 21850 when this tutorial was written)

In [ ]:
### [SOLUTION Q2]

adql = """
SELECT COUNT(*) FROM basic WHERE otype = 'C*'
"""  # we can apply functions to a column (here count)

In [ ]:
# Q3 - What are the most frequent spectral types for carbon stars? (It shouldn't be too surprising)

In [ ]:
### [SOLUTION Q3]

adql = """
SELECT COUNT(*) AS count_sp_type, sp_type
FROM basic WHERE otype = 'C*' GROUP BY sp_type
ORDER BY count_sp_type DESC
"""  # note, we used an alias here for the first time

In [ ]:
# Q4 - Get the definition of the object type `C*` from the description of objects types (`otypedef`) table

In [ ]:
### [SOLUTION Q4]

adql = """
SELECT * FROM otypedef WHERE otype = 'C*'
"""  # note that the hierarchy of object types is given in the `path` column

## Step #2: Using the `ipyaladin` widget

[![Aladin](https://custom-icon-badges.demolab.com/badge/Aladin-gray.svg?logo=aladin&logoColor=purple&logoWidth=20)](https://aladin.cds.unistra.fr/aladin.gml "https://aladin.cds.unistra.fr/aladin.gml") 

---

Let's now visualise the source on the Aladin sky atlas. 

![aladin](images/aladin.png)

The suite has three flavors: 

- **Aladin desktop**. It is the most complete version and offers a complete graphic interface to find catalogs and HiPS surveys and manipulate them (cutouts, operations, stacks...).
- **Aladin Lite**. Is a lighter version that runs in the browser. With the advance of browser technologies, it is gaining more and more functionalities every day.
- **iPyAladin**. It is a port of Aladin Lite in python notebooks. This is the flavor we'll use here.




### Basic usage of iPyAladin

We now open the `ipyaladin` widget. 
The next cell will open a new jupyter tab that you can then drag and drop on the side of the screen.


In [ ]:
# Create an instance of the Aladin class
aladin = Aladin(full_screen=True, background_color="white")
with Sidecar(title="ipyaladin", anchor="tab-after"):
    display(aladin)



Aladin's `target` property always contain the coordinates of the center of `iPyAladin`'s view.

In [ ]:
aladin.target

In [ ]:
aladin.target.ra  # we can access the ra and dec values easily

In [ ]:
## Exercise :

## Change the target of ipyaladin, then print the current value of `aladin.target`



As with any instance of Aladin Lite, you can interact with this widget.
* To zoom in and out, place your mouse pointer on the image and scroll.
* You can left-click on a point to grab it and center the reticule on it.
* If you would like to look at another target, you can use the search field search to get there.
* With layers you can select other image surveys.

The SIMBAD tool is really useful to know what we're looking at.

![simbad-pointer](images/simbad-pointer.png)

In [ ]:
## Exercise :

## Use the SIMBAD pointer tool to identify any object in the view


Let's now center the view on our star of interest. The target can also be set programmatically with either an object's name or with an astropy `SkyCoord` object:

In [ ]:
aladin.target = "CW Leo"

We can now change the field of view either by scrolling with the mouse or by running the next cell:

In [ ]:
aladin.fov = Angle(1.5, "arcmin")

In [ ]:
## Exercise :

## Edit the target or the field of view programmatically


### Layering surveys and HiPS

The images displayed by Aladin follow the [Hierarchical Progressive Survey (HiPS) standard](https://ivoa.net/documents/HiPS/20170519/index.html), in which a folder corresponds to a [Hierarchical, Equal Area, and iso-Latitude Pixelisation (HEALPix)](https://arxiv.org/abs/astro-ph/9905275) order, see figure II-3 b). At all orders, the images have the same number of pixels (here 512*512), but they represent a different sky fraction.

More information on [HiPS in Aladin](https://aladin.cds.unistra.fr/hips/).

![healpix_and_hips](images/healpix_and_hips.png)

On the default survey (the Digitized Sky Survey, DSS), the star and its surrounding environment appear as a faint red dot.


Let's display the HealPix grid and zoom in and out a little. If you do it very fast, you'll see the images appearing while you zoom in and `iPyAladin` retrieves the tiles from the HiPS.

![healpix-grid](images/activating-the-healpix-grid.png)


To find more HiPS, in this region, we can use `Aladin Lite`'s browse menu (see figure II-5.)

![browse-hips](images/aladin-browse-hips.png)



In [ ]:
## Exercise :

## Select and display another HIPS survey to the current view.


You can find the list of available sky maps at [https://aladin.cds.unistra.fr/hips/list](https://aladin.cds.unistra.fr/hips/list) .

Now that we have loaded new surveys, we change their appearance in the stack menu (as display in Figure II-6) or programmatically (see below). 

![layers-options](images/surveys-options.png)


### Retrieving a FITS cutout 

From this ipyaladin view, we can retrieve a cutout of the base layer from the server with the method `get_view_as_fits` which calls an other CDS service : `HiPS2Fits`. 

In [ ]:
# Go back to our previous target
aladin.target = "CW Leo"

# Replace the base survey programmatically
aladin.survey = "CDS/P/DECaLS/DR5/r"

In [ ]:
# Save the base layer as a FITS
dust_cloud = aladin.get_view_as_fits()

# Display the header (for information)
dust_cloud[0].header

The header contains information about the HiPS from which we did the cutout. Let's plot this FITS image with the WCS that corresponds to ipyaladin's view at the moment when we executed the `get_view_as_fits` method:

In [ ]:
# Save the WCS ((World Coordinate Systems) from the header
wcs = WCS(dust_cloud[0].header)

# Plot the saved FITS image using plt
ax = plt.subplot(projection=wcs)
plt.imshow(dust_cloud[0].data, cmap="binary_r", norm="asinh", vmin=0.0001);

In [ ]:
# Save the FITS image
dust_cloud.writeto("my_fits.fits")

And voilà, we can work on this FITS object like any other FITS opened in astropy.
    
Note that HiPS2FITS is also directly available as an astroquery module, without ipyaladin. To learn more about its use, see https://astroquery.readthedocs.io/en/latest/hips2fits/hips2fits.html

If you want to make your own HiPS so that they can be displayed in Aladin or any other HiPS viewer, [here](https://aladin.cds.unistra.fr/tutorials/OV-France-2024/) is a nice tutorial to do so.

### Adding FITS images to the ipyaladin view

We can also display FITS images directly in `iPyAladin`.

Let's access a file from VizieR:

In [ ]:
aladin.add_fits(
    "https://cdsarc.cds.unistra.fr/saadavizier/download?oid=864976546211823617",
    name="C2_H1",
)

Note that this method also works with a path to a FITS file. You can also drag and drop any FITS file from your computer directly into the `iPyAladin` window.

In [ ]:
## Exercise:

## Play with the FITS image options until the filaments are really visible


You can learn more on the features of the ipyAladin module here: [https://github.com/cds-astro/ipyaladin](https://github.com/cds-astro/ipyaladin) .

### Advanced section: the VizieR obscore service

We just used an image from VizieR. We could have found it by exploring [VizieR ObsCore](https://cdsarc.cds.unistra.fr/assocdata/) service.

ObsCore is a [VO standard](https://ivoa.net/documents/ObsCore/20170509/index.html) that constrains how observation data should be stored in a table. 
For example, you will see that ObsCore services will always have the same columns names. This allows to write ADQL queries the same way if we work with the Obscore service of VizieR or of an another data center !

Here are some of these columns -- go to the standard's page to get the other ones -- :

| column_name      | description                                                                                      |
|------------------|--------------------------------------------------------------------------------------------------|
| dataproduct_type | can be `image`, `cube`, `spectrum`, `sed`, `timeseries`, `visibility`, `event` or `measurements` |
| target_name      | astronomical object observed                                                                     |
| access_url       | URL to download the data                                                                         |
| s_ra             | central right ascension, ICRS                                                                    |
| s_dec            | central declinaison, ICRS                                                                        |
| instrument_name  | Name of the instrument                                                                           |
|        ...       | ...                                                                                              |


This is service is also available through VizieR's webpages, see figure II-6.

![obscore](images/obscore.png)

To find VizieR's ObsCore access URL, we can print the list of all existing ObsCore services in the VO:

In [ ]:
obscore_services = registry.search(registry.Datamodel("obscore"))
obscore_services.to_table().show_in_notebook(
    auto_fit_columns=True, selection_mode="cell"
)

In [ ]:
vizier_obscore = TAPService("https://cdsarc.cds.unistra.fr/saadavizier.tap/tap")

In [ ]:
cw_leo_skycoord = SkyCoord.from_name("CW Leo")
# this is how we do a cone search in ADQL. Again, don't worry, there's a whole lecture on ADQL later
# here this query says, please get me all the columns in the obscore table that are in 0.2° around CW Leo and
# that have been observed by ALMA
obscore_result = vizier_obscore.search(f"""SELECT * FROM obscore 
WHERE DISTANCE(POINT('', s_ra, s_dec), POINT('', {cw_leo_skycoord.ra.deg}, {cw_leo_skycoord.dec.deg})) < 0.2
AND facility_name = 'ALMA'""")
obscore_result.to_table().show_in_notebook(auto_fit_columns=True, selection_mode="cell")

In [ ]:
## Exercise:

## Add an other molecule to the current view an play with the opacity of the FITS image to see the difference with the one we selected above.


## Step #3: Catalogues manipulation with VizieR and X-match

[![Simbad](https://custom-icon-badges.demolab.com/badge/Simbad-gray.svg?logo=simbad&logoColor=lightblue&logoWidth=20)](https://simbad.unistra.fr/simbad/ "https://simbad.unistra.fr/simbad/")
[![Aladin](https://custom-icon-badges.demolab.com/badge/Aladin-gray.svg?logo=aladin&logoColor=purple&logoWidth=20)](https://aladin.cds.unistra.fr/aladin.gml "https://aladin.cds.unistra.fr/aladin.gml") 
[![Vizier](https://custom-icon-badges.demolab.com/badge/Vizier-gray.svg?logo=vizier&logoColor=orange&logoWidth=20)](https://vizier.cds.unistra.fr/viz-bin/VizieR "https://vizier.cds.unistra.fr/viz-bin/VizieR") 
[![Xmatch](https://custom-icon-badges.demolab.com/badge/Xmatch-gray.svg?logo=xmatch&logoColor=blue&logoWidth=20)](http://cdsxmatch.cds.unistra.fr/ "http://cdsxmatch.cds.unistra.fr/")


---

### Introduction 

The remaining of the tutorial is inspired by a work from Ripoche and collaborators, where they used "Carbon stars as standard candles":
* Ripoche et al. 2020, [2020MNRAS.495.2858R](https://simbad.cds.unistra.fr/simbad/sim-ref?bibcode=2020MNRAS.495.2858R) (Paper I)
* Parada et al. 2021, [2021MNRAS.501..933P](https://simbad.cds.unistra.fr/simbad/sim-ref?bibcode=2021MNRAS.501..933P) (Paper II)
* Parada et al. 2023, [2023MNRAS.522..195P](https://simbad.cds.unistra.fr/simbad/sim-ref?bibcode=2023MNRAS.522..195P) (Paper III)


Their idea is to derive a carbon-star luminosity function that will eventually be used to determine distances to galaxies at 50-60 Mpc and hence yield a value of the Hubble constant. 


To do, they focused primarly on the Large and Small Magellanic clouds (LMC and SMC). Later they expanded their sample with Local group galaxies (eg. NGC 6822, IC 1613, Wolf-Lundmark-Melotte, NGC 3109). 



---

### Data selection / datasets

- To build their sample in Paper I, Ripoche et al. 2020 queried 2MASS over a polygon centered on the LMC and SMC.
- They removed the foreground contamination from Milky Way stars, by cross-matching the 2MASS stars with those found in Gaia DR2.
- They used proper motions to select stars considered to be LMC and SMC members.
- Finally, they corrected the apparent magnitudes for extinction and reddening, and included the true distance moduli to these galaxies.

For computational purpose, we won't follow exactly the same steps as described above, but we will get to a similar result: a sample of carbon stars centered on the SMC. 
As an exercise, you can try later on to reproduce the results for the LMC.


#### Query Simbad

First, we want to select from the Simbad database all the carbon stars (objects with object_type = C*), 
along with their corresponding J and K magnitudes. 
These two sets of magnitudes are important as they will be used later for a color selection.


In SIMBAD, the information is organized in [different tables](https://simbad.cds.unistra.fr/simbad/tap/tapsearch.html). The most powerful way to retrieve information from them is to use ADQL (Astronomical Data Query Language) in the `query_tap` method. You'll learn more about ADQL in another tutorial of this school.


To do so, we need to join two tables:
* `basic`: contains information like the main identifier, coordinates, proper motions, radial velocities, ...
* `allfluxes`: contains a quick view of the fluxes measurements (a more complete view with errors and bibcode in which the information was taken is in the table `flux`). 

![allfluxes](images/allfluxes.png)


In [ ]:
# Create an instance of the Simbad class
simbad = Simbad()

# This is an ADQL query
# Select all the objects with object_type = C* where the coordinates, the J and the K magnitudes are not missing
simbad_carbon_stars = simbad.query_tap(
    """
    SELECT main_id, ra, dec, J, K
    FROM basic JOIN allfluxes ON oid = oidref
    WHERE otype='C*'
    AND ra IS NOT NULL
    AND J IS NOT NULL
    AND K IS NOT NULL
    """,
    maxrec=100000,
)

# Print the size of the sample
print(
    f"\n[Info] From the previous TAP query, we get a sample of C* stars from Simbad with: {len(simbad_carbon_stars)} objects.\n"
)


# We define the "J-K" column that will be used later on
simbad_carbon_stars["J_minus_K"] = simbad_carbon_stars["J"] - simbad_carbon_stars["K"]


# Show a subset of the resulting table
simbad_carbon_stars[0:10]

##### Launch Aladin

In [ ]:
# Launch Aladin instance (we create a new and clear instance)
aladin = Aladin(full_screen=True, background_color="white")
with Sidecar(title="ipyaladin", anchor="tab-after"):
    display(aladin)

In [ ]:
# Plot a polygon around SMC in Aladin
aladin.target = SkyCoord(13, -73, unit="deg")  # Position around SMC
aladin.fov = Angle(6, "deg")  # Change Field-of-view
aladin.rotation = Angle(0, "deg")  # Change rotation
aladin.survey = "DSS/DSSColor"

# Draw a polygon around the SMC, following Ripoche et al. 2020
aladin.add_graphic_overlay_from_stcs(
    "Polygon ICRS 7.78 -74.15, 18.30 -74.15, 18.30 -71.43, 7.78 -71.43",
    color="purple",
    name="polygon-around-SMC",
)

Note: The [STC-S notation for shapes is a VO draft standard](https://ivoa.net/documents/STC-S/20130917/index.html). You'll also encounter it in [ADQL](https://ivoa.net/documents/ADQL/20231215/index.html) and [ObsCore](https://ivoa.net/documents/ObsCore/20170509/index.html) queries.

Let's display again the HealPix grid! 

#### Multi-order coverage (MOC) maps

The [Multi-Order Coverage (MOC)](https://www.ivoa.net/documents/MOC/20220727/index.html) is another standard based on the HEALPix grid. A space MOC describes a region of the sky as an ensemble of HEALPix cells of different orders.

![moc](images/SMOC.png)

##### Visualisation MOC

Let's now build a MOC corresponding to this polygon. We first need to chose an order corresponding to a reasonable resolution. Let's say 0.5°:

In [ ]:
# Choose the order corresponding to resolution 0.5 deg
order_res_05 = MOC.spatial_resolution_to_order(Angle(0.5, "deg"))
order_res_05

In [ ]:
# Plot the MOC that will include all the purple polygon
MOC_SMC = MOC.from_polygon_skycoord(
    SkyCoord(
        [[7.78, -74.15], [18.30, -74.15], [18.30, -71.43], [7.78, -71.43]], unit="deg"
    ),
    max_depth=order_res_05,
)

# Add the MOC to Aladin
aladin.add_moc(MOC_SMC, color="teal", name="SMC-moc-polygon")

You'll see that the MOC is an approximated representation of the polygon. The more precise the order chosen, the closer to the polygon the contours are. The corners of the given shape will always be included in the MOC.

In [ ]:
### Exercise :

## - Change the order of the MOC (reminder, MOCs orders are comprised between 0 and 29),
## - Check the corresponding precision (with `order_to_spatial_resolution`)
## - You can do: order.to("deg"), to transform the units from Radian to Degrees
## - Display it in ipyaladin -- with your favourite color


##### Selection by MOC


In [ ]:
# We can also visualize the list of carbon stars that we got from SIMBAD
aladin.add_table(simbad_carbon_stars, color="lightblue", name="simbad")

The next step will be then to select only the carbon stars that are within the MOC polygon.

In [ ]:
# Create a mask to select only the "SMC" stars
mask_smc = MOC_SMC.contains_skycoords(
    SkyCoord(simbad_carbon_stars["ra"], simbad_carbon_stars["dec"], unit="deg")
)
simbad_carbon_stars_smc = simbad_carbon_stars[mask_smc]

# Add the sub-sample to Aladin for visualisation
aladin.add_table(simbad_carbon_stars_smc, color="lightpink", name="simbad_smc")

# Print the size of the sample
print(
    f"[Info] The sub-sample centered around the SMC contains {len(simbad_carbon_stars_smc)} stars."
)

#### Advanced section: Building a MOC around an object

*This section won't be played in the live tutorial*

Let's build a Space-MOC around `CW Leo` (object from Step #1). We can either do it with ipyaladin's graphic interface or programmatically.

![MOC-in_ipyaladin](images/MOC-in-ipyaladin.png)

Let's open the MOC we built with Aladin's interface:

In [ ]:
with open("cone.json") as moc_file:
    moc_from_aladin_interface = MOC.from_json(json.load(moc_file))

and add it back to the view

In [ ]:
aladin.add_moc(moc_from_aladin_interface, color="yellow", name="MOC_from_file_system")

Or we can create the MOC programmatically for more control over its properties. To chose the MOC's precision, we can use one of MOCPy's helper methods:

In [ ]:
chosen_order = MOC.spatial_resolution_to_order(Angle(1, "arcsec"))
chosen_order

This means that to get a MOC with a precision of at least 1", we have to build a MOC with its smallest cells being of order 18.

In [ ]:
cw_leo_coordinates = SkyCoord.from_name("CW Leo")  # this calls the SESAME CDS service

moc_around_star = MOC.from_cone(
    cw_leo_coordinates.ra,
    cw_leo_coordinates.dec,
    radius=Angle(4, "arcsec"),
    max_depth=chosen_order,
)

Let's display this in the `ipyaladin` view:

In [ ]:
aladin.add_moc(
    moc_around_star,
    name="moc_around_star",
    color="lightpink",
    opacity=0.5,
    fill=True,
    edge=True,
)

The `name="moc_around_star"` we chose here will be the one appearing in the `Stack` menu:

![stack_name](images/name_of_moc.png)

#### Advanced exercise

You can reproduce a similar result for the LMC stars.

In [ ]:
### Exercise : same manipulation with LMC

### Some tips

## Rough position for LMC
# aladin.target = SkyCoord(81, -70, unit="deg")

## Coordinates for the polygon around LMC
# polygon(65 -76, 65 -62, 100 -62, 100 -76)


### Plots

We now have a sample of ~ 2000 "SMC"-carbon stars, that we will use to compute the color-magnitude diagrams.
Please note that we did not do any careful membership analysis above, but it is sufficient for the purpose of this tutorial.

#### 1 - Apparent magnitude

In [ ]:
# Plot: apparent magnitude

fig, ax = plt.subplots(figsize=(10, 8))

# All carbon stars
ax.scatter(
    simbad_carbon_stars["J_minus_K"],
    simbad_carbon_stars["J"],
    marker=".",
    label="All sample",
)

# Carbon stars from the SMC
ax.scatter(
    simbad_carbon_stars_smc["J_minus_K"],
    simbad_carbon_stars_smc["J"],
    marker=".",
    color="red",
    label="SMC",
)


# Add a rectangle with the data that we will select for the exercice
ax.add_patch(Rectangle((1, 17), 2, 5, linewidth=1, edgecolor="g", facecolor="none"))

ax.set_xlabel("$(J - K_s)$", fontsize=16)
ax.set_ylabel("$J$", fontsize=16)

ax.set_xlim(-0.5, 5)
ax.set_ylim(23, 0)

ax.legend(loc="upper right");

In [ ]:
#### Exercise (if time allows):

# From the previous color-magnitude diagram, select a sub-sample of stars (green rectangular) and locate them in Aladin.
# Criteria: J > 17 and 1 < (J-K) < 3


In [ ]:
# [SOLUTION] Exercise: Select a sub-sample of stars and plot them again in the color-magnitude diagram
# Criteria: J > 17 and 1 < (J-K) < 3


# Selection of the stars
index_jk = (
    (simbad_carbon_stars["J"] > 17)
    & (simbad_carbon_stars["J_minus_K"] > 1)
    & (simbad_carbon_stars["J_minus_K"] < 3)
)


# We now apply these selection criteria to our table of point sources.
simbad_extra = simbad_carbon_stars[index_jk]
print(f"\n[Info] We have selected: {len(simbad_extra)} stars all over the sky.\n")

# Add them to Aladin for display
aladin.add_table(simbad_extra, color="green", name="sub_selection")

Zoom in and explore the iPyAladin's view to see if the new subset of stars in green has a particularity.


#### Selecting objects in Aladin

You can also select a sub-sample of the tables currently visible in the iPyAladin's view. See figure III-5.

![source_selection](images/source_selection.png)

In [ ]:
aladin.selected_objects

In [ ]:
## Exercise:

## Add this sub-selection of stars to the previous color-magnitude diagram and see if they fall within a specific region
## Note: You will need to transform the "J" and "J_minus_K" columns from string to float before plotting: eg. new_sample["J"] = list(map(float, new_sample["J"]))


#### 2 - Absolute magnitude

We now have to transform apparent magnitudes to absolute magnitudes. 
The first step is to include the mean distance modulus to the SMC, and then we need to correct for extinction (magnitude) and reddening (colour).

We follow the equations 4-7 and Table 3 of [Ripoche et al. 2020](https://academic.oup.com/mnras/article/495/3/2858/5837581?login=false). 

In [ ]:
# Transformation from apparent to absolute magnitudes

# Values for SMC
k_J = 0.131
k_Ks = 0.016
k_B = 1.374
coeff_Av_SMC = k_J / (k_B - 1)
coeff_Av_SMC_color = (k_J - k_Ks) / (k_B - 1)
mu_SMC = 18.96
E_BV_SMC = 0.084

# DEfine two new columns in the table
simbad_carbon_stars_smc["M_J"] = (
    simbad_carbon_stars_smc["J"] - mu_SMC - (coeff_Av_SMC * E_BV_SMC)
)
simbad_carbon_stars_smc["JK_0"] = simbad_carbon_stars_smc["J_minus_K"] - (
    coeff_Av_SMC_color * E_BV_SMC
)

In [ ]:
# Plot: absolute magnitude
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(
    simbad_carbon_stars_smc["JK_0"],
    simbad_carbon_stars_smc["M_J"],
    marker=".",
)

# Add the vertical lines for the Carbon stars color selection (Ripoche+2020): 1.4 < (J-K) < 2
ax.axvspan(1.4, 2, edgecolor="orange", fill=False, linestyle="dashdot")

# Add the x-y labels
ax.set_xlabel("$(J - Ks)_0$", fontsize=16)
ax.set_ylabel("$M_J$", fontsize=16)

# Set limits axis
ax.set_xlim(-0.5, 3)
ax.set_ylim(-3, -10);

Based on the horizontal feature in the CMD of the SMC, Ripoche et al. 2020 built a colour selection such that: $1.4 < (J − K_s)_0 < 2 $   (cf orange vertical lines in the figure above), to separate carbon-stars from oxygen-rich AGB. 


#### 3 - Carbon-star luminosity function (CLSF)

We are going to derive the CSLF of the SMC for all the stars within the previous colour range.

In [ ]:
# Colour selection
index_color = (simbad_carbon_stars_smc["JK_0"] > 1.4) & (
    simbad_carbon_stars_smc["JK_0"] < 2
)

# We now apply these selection criteria to our table of point sources.
smc_short_sample = simbad_carbon_stars_smc[index_color]

# Print the size of the sample
print(
    f"\n[Info] With the previous colour section, our final sample now contains {len(smc_short_sample)} stars.\n"
)

Following Ripoche et al., we derived the J-band CSLF of the SMC for all the stars within the colour range.

In [ ]:
# Plot: probability density function
fig, ax = plt.subplots(figsize=(10, 8))

# Number of bins set to 50:
label_simbad = f"Simbad stars SMC ({len(smc_short_sample)} stars)"
ax.hist(smc_short_sample["M_J"], 50, histtype="step", label=label_simbad, density=True)

ax.set_xlabel("$M_J$", fontsize=16)
ax.set_ylabel("probability density function (PDF)", fontsize=16)

ax.set_xlim(-3, -9)

plt.legend(loc="upper right");

![fig8_ripoche](images/fig8_Ripoche.png)

On figure III-6, (figure 8 from Ripoche et al. 2020), two datasets appear: one from 2MASS and another from Raimondo et al. 2005 ([2005A&A...438..521R](https://simbad.cds.unistra.fr/simbad/sim-ref?bibcode=2005A%26A...438..521R)).

The spectroscopic catalogue of 1073 carbon stars in the SMC by Raimondo et al. (2005) was introduced by Ripoche et al. to confirm the nature of the stars that appear in this horizontal feature, and thus validate their colour selection.

In the remaining of this tutorial, we will follow the authors'steps by cross-matching this sample from Raimondo with 2MASS in order to add those carbon stars to our existing dataset.



#### Extra exercise

You can reproduce the same plots (apparent and absolute magnitude, carbon-star luminosity function) for the LMC stars.

###  Catalogues manipulation with VizieR and X-match


#### Query VizieR

VizieR is a curated collection of astronomical catalogues. Each catalogue is either accompanying a publication or is a curated version (columns metadata and explanations are normalized) of a public survey.

![vizier](images/vizier.png)

In [ ]:
# Create an instance of the VizieR class
vizier = Vizier(row_limit=1)

Here we limit the VizieR search to `1` row. Note that by default VizieR will only return `50` entries (as in the web interface). To get all sources, set `row_limit=-1`.

A good practice is to check first what data are available by querying only a subsample, then customize your Vizier query with the useful rows and columns before downloading an entire catalog.

##### Sample 1: 2MASS catalogue


In [ ]:
# Let's find VizieR's identifier for `2MASS`:
catalogs_2MASS = vizier.find_catalogs("2MASS")

# Print the found catalogs
for key, value in catalogs_2MASS.items():
    print(key, value.description)

In the following, we will use the "2MASS All-Sky Catalog of Point Sources" catalogue: VizieR name = 'II/246'.

In [ ]:
# Check the available tables for 2MASS
cat_2MASS_PSC = vizier.get_catalogs("II/246")

print(cat_2MASS_PSC)

# There is a single table in the table list
cat_2MASS_PSC = cat_2MASS_PSC[0]

In [ ]:
# Print some information from the table
for name, column in cat_2MASS_PSC.columns.items():
    print(f"{name}: {column.description}")

##### Sample 2: Raimondo et al. 2005 catalogue


In [ ]:
# Get the names of the tables from Raimondo et al. 2005
cat_Raimondo = Vizier(row_limit=-1).get_catalogs("2005A&A...438..521R")

# Print the metadata
cat_Raimondo[0].meta

In [ ]:
# Get some information
Vizier(catalog="J/A+A/438/521").get_catalog_metadata()

In [ ]:
# Select the only table available
tab_Raimondo = cat_Raimondo[0]

# Print the size of the table
print(f"\n [Info] Raimondo+05 catalogue contains {len(tab_Raimondo)} stars.\n")

# Print the first rows
tab_Raimondo[0:10]

In [ ]:
# Add the Raimondo stars to Aladin
aladin.add_table(
    tab_Raimondo, shape="circle", name="Raimondo", source_size=15, color="orange"
)

We see that most of the stars in Raimondo seem to correspond to a star in SIMBAD (there is a pink cross in the middle of most orange circles) but not all of them!

##### Equivalent TAP query
*Note:* We could also have written a TAP/ADQL query to get the same catalog from VizieR (see below).

In [ ]:
# Define the TAP service
tap_vizier = TAPService("https://tapvizier.cds.unistra.fr/TAPVizieR/tap")

# Create the query
new_query = 'SELECT MACHO, RAJ2000, DEJ2000, Bmag, Rmag FROM "J/A+A/438/521/Cstars"'


cat_R05 = tap_vizier.search(new_query).to_table()
cat_R05[0:10]

#### X-match service

We have now identified the two catalogues we want to cross match: Raimondo et al. 2005 (~ 1100 objects) and 2MASS from Cutri et al. 2003 (~ 470 million objects)! 

We use the X-Match python module for cross-identification of objects. It is particularly efficient and fast with very large catalogs, like 2MASS / AllWISE. All we need are the names of the catalogs, the names of the columns containing the coordinates of the sources, and the desired maximum distance for the match (optionally the area as well, otherwise all-sky is the default).

![xmatch](images/x-match.png)


In [ ]:
# Create an instance of the XMatch class
xmatch = XMatch()
xmatch.URL = "https://cdsxmatch.cds.unistra.fr/xmatch/api/v1/sync"

We can display its help page with:

In [ ]:
xmatch.query?

In [ ]:
tab_Raimondo.meta["name"]

In [ ]:
# Launch the cross-match between Raimondo et al. (2005) and 2MASS
cross_matched = xmatch.query(
    cat1=tab_Raimondo.meta["name"],  # Name of the catalogue
    cat2="II/246/out",  # name of 2MASS
    max_distance=Angle(1.5, "arcsec"),
)

# Print the first rows
cross_matched[0:10]

Before moving on to the plots, let's make sure there is only one match per star from the input catalogue.

In [ ]:
print(f"""\n[Info] The cross-matched table between Raimondo+05 and 2MASS contains: {len(cross_matched["MACHO"])} stars,
and {len(set(cross_matched["MACHO"]))} unique MACHO identifiers.\n""")

#### Plots

##### Plot 1: Apparent magnitude

With our cross-matched sample (and the near-infrared photometry), we can now reproduce again the CMD plots.

In [ ]:
### Exercise: Add the cross_matched sample to the previous apparent magnitude plot

In [ ]:
### [SOLUTION] Exercise: Add the cross_matched sample to the previous apparent magnitude plot

fig, ax = plt.subplots(figsize=(10, 8))

# All carbon stars
ax.scatter(
    simbad_carbon_stars["J_minus_K"],
    simbad_carbon_stars["J"],
    marker=".",
    label="All sample",
)

# Carbon stars from the SMC
ax.scatter(
    simbad_carbon_stars_smc["J_minus_K"],
    simbad_carbon_stars_smc["J"],
    marker=".",
    color="red",
    label="SMC",
)

# Carbon stars from Raimondo+05
ax.scatter(
    cross_matched["Jmag"] - cross_matched["Kmag"],
    cross_matched["Jmag"],
    marker="v",
    s=4,
    color="orange",
    label="Raimondo+05",
)

ax.set_xlabel("$(J - K_s)$", fontsize=16)
ax.set_ylabel("$J$", fontsize=16)

ax.set_xlim(-0.5, 5)
ax.set_ylim(23, 0)

plt.legend(loc="upper right");

##### Plots 2-3: Absolute magnitude + CSLF

In [ ]:
# Transformation from apparent to absolute magnitudes
cross_matched["M_J"] = cross_matched["Jmag"] - mu_SMC - (coeff_Av_SMC * E_BV_SMC)
cross_matched["JK_0"] = (cross_matched["Jmag"] - cross_matched["Kmag"]) - (
    coeff_Av_SMC_color * E_BV_SMC
)

# Colour selection
index_color_r05 = (cross_matched["JK_0"] > 1.4) & (cross_matched["JK_0"] < 2)

# We now apply these selection criteria to our table of point sources.
raimondo_short_sample = cross_matched[index_color_r05]

# Print the size of the sample
print(
    "\n[Info] With the Ripoche et al. 2020 colour section, our final sample now"
    f" contains {len(raimondo_short_sample)} stars.\n"
)

In [ ]:
# Plots: absolute magnitude + CSLF
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8))
fig.suptitle("Plots for SMC stars")


################

## 1st plot
ax1.scatter(
    simbad_carbon_stars_smc["JK_0"],
    simbad_carbon_stars_smc["M_J"],
    marker=".",
    s=6,
    label="Simbad stars SMC",
)

ax1.scatter(
    cross_matched["JK_0"],
    cross_matched["M_J"],
    marker="v",
    s=6,
    color="orange",
    label="Raimondo+05",
)

# Add the vertical lines for the Carbon stars color selection (Ripoche+2020): 1.4 < (J-K) < 2
ax1.axvspan(1.4, 2, edgecolor="orange", fill=False, linestyle="dashdot")

ax1.set_xlabel("$(J - K_s)_0$", fontsize=16)
ax1.set_ylabel("$M_J$", fontsize=16)

ax1.set_xlim(0, 3)
ax1.set_ylim(-3, -10)


################

# Second plot: probability density function

# label_simbad = f"Simbad stars SMC ({len(smc_short_sample)} stars)"
ax2.hist(smc_short_sample["M_J"], 50, histtype="step", density=True)

# Number of bins set to 50:
# label_raimondo = f"Raimondo+05 ({len(raimondo_short_sample)} stars)"
ax2.hist(
    raimondo_short_sample["M_J"],
    25,
    histtype="step",
    linestyle=("dashed"),
    density=True,
)

# Add the labels
ax2.set_xlabel("$M_J$", fontsize=16)
ax2.set_ylabel("probability density function (PDF)", fontsize=16)

ax2.set_xlim(-4, -8)


################

# Add the legend
ax1.legend(loc="upper left");

#### Extra exercise
You can reproduce a similar result for the LMC stars.

In [ ]:
# [TIPS LMC]
# cross-matching 2MASS with the LMC carbon stars from Kontizas et al. 2001,
# Bibcode = 2001A&A...369..932K
# VizieR table: `J/A+A/369/932` .

#### Another X-match exercise (if time allows)

Earlier, we mentioned that most of the stars in Raimondo (orange circle in Aladin) seem to correspond to a star in SIMBAD (lightpink in Aladin). Let's verify it!

We can also see the connection between Raimondo and Simbad by doing a cross-match between our cross-matched catalogue (Raimondo et al. (2005) - 2MASS) on one side and the `SIMBAD` database on the other side.

The version of SIMBAD accessible *via* the X-Match service is a flatten version of the SIMBAD database where a fixed set of columns have been pre-selected. You will for example see that all the fluxes and all the columns of `basic` will be present in the result.


In [ ]:
# Launch the cross-match between Raimondo et al. (2005) - 2MASS [cross_matched] and Simbad
cross_matched_with_simbad = xmatch.query(
    cat1=cross_matched,
    cat2="simbad",
    max_distance=Angle(1.5, "arcsec"),
    colRA1="RAJ2000",
    colDec1="DEJ2000",
)

# Print the first rows
cross_matched_with_simbad[0:10]

##### Deal with the duplicates


In [ ]:
# Print the size of the cross-matched table
print(
    f"\n[Info] The cross-matched table between Raimondo+05 - 2MASS and Simbad contains: {len(cross_matched_with_simbad['MACHO'])} stars, and {len(set(cross_matched_with_simbad['MACHO']))} unique identifiers.\n"
)

In [ ]:
# Let's group the stars by identifier
counter = Counter(cross_matched_with_simbad["MACHO"])
counter.most_common()[0:10]

We can see that there are three sets of objects with 2 identifiers. Let's focus on one of them.

In [ ]:
# Check the multiple matches
multiple_match = cross_matched_with_simbad[
    cross_matched_with_simbad["MACHO"] == "211.16418.2"
]
multiple_match

The closest match corresponds to the same object in Simbad (`RAW 1030`), a carbon star, while the second match corresponds to an AGN candidate (`XMMU J005711.4-731316`). Therefore it is important to check the distances (angDist) between the objetcts, but also the object types.

In [ ]:
## Exercise

# You can check the multiple matches for the two other cases.

#### Conclusion article

We can see that the luminosity functions of the stars from our Simbad query and the ones from the Raimondo catalogue correlate very well, in agreement with Figure 8 from Ripoche et al. 2020.

From these plots, the authors used maximum-likelihood statistics in order to estimate the median absolute magnitude 
and the intrinsic magnitude dispersion σ of the carbon stars in the SMC. 
They obtained the following median absolute magnitude:

* 2MASS sample: -6.160
* Raimondo catalogue: -6.172,
which is consistent with what we get.

Ripoche and collaborators developed a method, based purely on two near-infrared bands, to identify carbon stars in the MCs. This method relies on an accurate measurement of the distance to the LMC (first rung of the cosmic distance ladder), and the SMC. Their method uses only on a selection within a (J − Ks)0 colour range, but is affected by metallicity.

Carbon stars are brighter than the TRGB. They are also more luminous than Cepheids and require only a single photometric measurement to establish their luminosity. Consequently, thanks to the next generation of telescopes (JWST, ELT, and TMT), carbon stars could be detected in MC-type galaxies at distances of about 50–60 Mpc. Those distances are large enough that the effects of the galaxies’ peculiar velocities are not important, yielding a two-step distance ladder to the scale of the Universe. Their final goal is to eventually improve the measurement of the Hubble constant and explore the current tensions surrounding its value.
 

## Conclusions

### Summary

We used the following services

- [![Simbad](https://custom-icon-badges.demolab.com/badge/Simbad-gray.svg?logo=simbad&logoColor=lightblue&logoWidth=20)](https://simbad.unistra.fr/simbad/ "https://simbad.unistra.fr/simbad/"): Query objects
- [![Aladin](https://custom-icon-badges.demolab.com/badge/Aladin-gray.svg?logo=aladin&logoColor=purple&logoWidth=20)](https://aladin.cds.unistra.fr/aladin.gml "https://aladin.cds.unistra.fr/aladin.gml"): Data visualisation and selection, HiPS, MOC
- [![Vizier](https://custom-icon-badges.demolab.com/badge/Vizier-gray.svg?logo=vizier&logoColor=orange&logoWidth=20)](https://vizier.cds.unistra.fr/viz-bin/VizieR "https://vizier.cds.unistra.fr/viz-bin/VizieR"): Catalogue manipulation
- [![Xmatch](https://custom-icon-badges.demolab.com/badge/Xmatch-gray.svg?logo=xmatch&logoColor=blue&logoWidth=20)](http://cdsxmatch.cds.unistra.fr/ "http://cdsxmatch.cds.unistra.fr/"): Cross-matching large catalogues




We saw the following VO standards

- [TAP](https://ivoa.net/Documents/TAP/20190927/index.html)/[ADQL](https://ivoa.net/Documents/ADQL/20231215/index.html) : allows to retrieve information from online tables with a standard query language.
- [HiPS](https://ivoa.net/documents/HiPS/20170519/index.html) : these hierarchical, progressively refined sky surveys are optimized to vizualise large surveys.
- [MOC](https://ivoa.net/documents/MOC/20220727/index.html) : the Multi-Order Coverages describe a complex region of the sky. We can perform operation on MOCs and retrieve only the sources of a catalog that fall within the MOC.
- [STC-S](https://ivoa.net/documents/STC-S/20130917/index.html) : is a way to write space-time coordinates as a standard string.
- ([ObsCore](https://ivoa.net/documents/ObsCore/20170509/index.html): in an advanced exercise) : the ObsCore data model allows to write the same queries on different providers by constraining the columns that should exist.

The use-case was on Carbon Stars, but feel free to come to any of the instructors with your own scientific questions to see if any of the ideas/services presented here could be useful for your research!

### Citing the CDS services

If any of the CDS services was useful for your research, please consider acknowledging them in your papers.
Here is the page with the citations instructions: https://cds.unistra.fr/help/acknowledgement/

### Useful links / Documentation

* Simbad astroquery module: [https://astroquery.readthedocs.io/en/latest/simbad/simbad.html](https://astroquery.readthedocs.io/en/latest/simbad/simbad.html)
* Vizier astroquery module: [https://astroquery.readthedocs.io/en/latest/vizier/vizier.html](https://astroquery.readthedocs.io/en/latest/vizier/vizier.html)
* Xmatch astroquery module: [https://astroquery.readthedocs.io/en/latest/xmatch/xmatch.html](https://astroquery.readthedocs.io/en/latest/xmatch/xmatch.html)
* Simbad tables: [https://simbad.cds.unistra.fr/simbad/tap/tapsearch.html](https://simbad.cds.unistra.fr/simbad/tap/tapsearch.html)
* ipyAladin module: [https://github.com/cds-astro/ipyaladin](https://github.com/cds-astro/ipyaladin)
* HiPS2FITS (as an astroquery module): [https://astroquery.readthedocs.io/en/latest/hips2fits/hips2fits.html](https://astroquery.readthedocs.io/en/latest/hips2fits/hips2fits.html)
* HiPS tutorial (to create your own HiPS): [https://aladin.cds.unistra.fr/tutorials/OV-France-2024/](https://aladin.cds.unistra.fr/tutorials/OV-France-2024/)
* List of Hierarchical Progressive Surveys provided by all public HiPS servers: [https://aladin.cds.unistra.fr/hips/list](https://aladin.cds.unistra.fr/hips/list)
